# 🎙️ Whisper Transcriber (Google Colab)

Transcribe **audio and video files stored in Google Drive** into text using OpenAI's Whisper.

## 📁 Folder layout in Google Drive
```
MyDrive/Notebooks/transcribeWhisper/
├── videos/       ← video files
├── sounds/       ← audio files
└── transcribes/  ← transcrip results
```


## 🚀 How to use (3 steps)
1. **Switch on the GPU** (much faster): *Runtime → Change runtime type → T4 GPU → Save*.
2. **Put your files in Google Drive** (see the folder layout below).
3. **Press the ▶ button** on each step below, from top to bottom. Wait until a step shows ✅ before starting the next one.
   Only the **Settings** step (Step 2) needs your input. All other steps just need ▶.

> Tip: after choosing your settings you can also use **Runtime → Run all**. Google will ask permission to access your Drive the first time — click *Allow*.


## 🧭 Workflow
| Steps |  |
|---|---|
| 1 | Installs the tools and connects Google Drive |
| 2 | Choose model, language and output formats |
| 3 | Converts videos to audio (if you have videos) |
| 4 | Load the Whisper model |
| 5 | Transcribe every audio file |
| 6 | Review the results |
| 7 | Frees memory |
| Tool A | Extracts audio from videos on **PC** |
| Tool B | Converts `.srt` → `.txt` — on PC **or** in Colab |

## 1. Setup

Install Whisper and `ffmpeg`, then mount Google Drive.

In [ ]:
#@title ▶ 1a. Install the tools { display-mode: "form" }
# Installs Whisper (the transcription engine) and ffmpeg (reads audio/video files).
print("⏳ Installing tools...")
!pip install -q -U openai-whisper
!apt-get -qq update && apt-get -qq install -y ffmpeg
print("✅ Tools installed.")

In [ ]:
#@title ▶ 1b. Connect Google Drive { display-mode: "form" }
# Will ask permission to access Drive. Click "Connect" / "Allow".
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive connected.")

## 2. Settings


| Setting | Detail |
|---|---|
| **BASE_DIR** | The folder in Google Drive that contains `videos`, `sounds` and `transcribes`. |
| **MODEL** | Bigger = more accurate but slower. `.en` models are English-only. `turbo` is fast and accurate for many languages. |
| **LANGUAGE** | The language spoken in your files, or `Auto-detect`. Choosing it yourself is faster and more reliable. |
| **TASK** | `transcribe` = text in the original language. `translate` = English text. |
| **INITIAL_PROMPT** | *(Optional)* Names or technical terms to help Whisper spell them correctly. |
| **SAVE_*** | Which files to create. `txt` is plain text; `srt`/`vtt` are subtitles with timestamps. |
| **SKIP_EXISTING** | Skip files that were already done, so you can safely run again. |

Model guide:
- `tiny` / `base` → quick drafts
- `small` / `medium` → good balance
- `large-v3` → best accuracy.

In [ ]:
#@title Settings { run: "auto" }

BASE_DIR = "/content/drive/MyDrive/Notebooks/transcribeWhisper"  #@param {type:"string"}

MODEL = "medium.en"  #@param ["tiny", "tiny.en", "base", "base.en", "small", "small.en", "medium", "medium.en", "large-v3", "turbo"]
LANGUAGE = "English"  #@param ["Auto-detect", "English", "Arabic", "Persian"]
TASK = "transcribe"  #@param ["transcribe", "translate"]
INITIAL_PROMPT = ""  #@param {type:"string"}

SAVE_TXT = True   #@param {type:"boolean"}
SAVE_SRT = False  #@param {type:"boolean"}
SAVE_VTT = False  #@param {type:"boolean"}
SAVE_JSON = False  #@param {type:"boolean"}

SKIP_EXISTING = True  #@param {type:"boolean"}

In [ ]:
#@title ▶ 2b. Apply and check the settings { display-mode: "form" }
# Turns your choices into what Whisper needs, creates missing folders and fixes common mistakes.
from pathlib import Path
import torch

# --- Language name -> Whisper language code (None = auto-detect) ---
LANGUAGE_CODES = {
    "Auto-detect": None, "English": "en", "Arabic": "ar", "Persian": "fa",
}
language_code = LANGUAGE_CODES[LANGUAGE]
model_name = MODEL

# English-only (.en) models cannot handle other languages or translation.
if model_name.endswith(".en") and (language_code != "en" or TASK == "translate"):
    fixed = model_name[:-3]
    print(f"⚠️  '{model_name}' only works for English transcription. Using '{fixed}' instead.")
    model_name = fixed

# The turbo model cannot translate.
if model_name == "turbo" and TASK == "translate":
    print("⚠️  'turbo' cannot translate. Using 'large-v3' instead.")
    model_name = "large-v3"

# Folders
base_dir = Path(BASE_DIR)
videos_dir = base_dir / "videos"
sounds_dir = base_dir / "sounds"
transcribes_dir = base_dir / "transcribes"

if not Path("/content/drive/MyDrive").exists():
    print("❌ Google Drive is not connected.")
else:
    for folder in (videos_dir, sounds_dir, transcribes_dir):
        folder.mkdir(parents=True, exist_ok=True)

    output_formats = [fmt for fmt, on in
                      (("txt", SAVE_TXT), ("srt", SAVE_SRT), ("vtt", SAVE_VTT), ("json", SAVE_JSON)) if on]
    if not output_formats:
        print("⚠️  No output format was selected. Using 'txt'.")
        output_formats = ["txt"]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("⚠️  No GPU found, so transcription will be slow.")
        print("    Fix: Runtime → Change runtime type → T4 GPU")

    print("✅ Settings applied:")
    print(f"   Model    : {model_name}")
    print(f"   Language : {LANGUAGE}")
    print(f"   Task     : {TASK}")
    print(f"   Formats  : {', '.join(output_formats)}")
    print(f"   Device   : {'GPU' if device == 'cuda' else 'CPU (slow)'}")
    print(f"   Folder   : {base_dir}")

## 3. Convert Videos to Audio *(optional)*

Every video in `videos/` is converted to a **mono, 16 kHz MP3** in `sounds/` .
Videos that were already converted are skipped.

In [ ]:
#@title ▶ 3. Convert videos to audio { display-mode: "form" }
import subprocess

if 'videos_dir' not in globals():
    print("❌ Please run the settings first.")
else:
    VIDEO_EXTENSIONS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v", ".flv", ".wmv"}
    video_files = sorted(p for p in videos_dir.iterdir() if p.suffix.lower() in VIDEO_EXTENSIONS)

    if not video_files:
        print(f"ℹ️  No videos found in: {videos_dir}")
        print("   Nothing to convert.")
    else:
        print(f"Found {len(video_files)} video(s).\n")
        for video_path in video_files:
            audio_path = sounds_dir / (video_path.stem + ".mp3")

            if audio_path.exists() and SKIP_EXISTING:
                print(f"⏭️  Already converted: {video_path.name}")
                continue

            print(f"🎬 Converting: {video_path.name}")
            command = [
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", str(video_path),
                "-vn",            # drop the picture, keep the sound
                "-ac", "1",       # mono
                "-ar", "16000",   # 16 kHz (what Whisper uses)
                "-b:a", "64k",
                str(audio_path),
            ]
            result = subprocess.run(command, capture_output=True, text=True)
            if result.returncode != 0:
                print(f"   ❌ Could not convert this file: {result.stderr.strip()}")
            else:
                print(f"   ✅ Saved as {audio_path.name}")
        print("\n✅ Video conversion finished.")

## 4. Load the Whisper Model


In [ ]:
#@title ▶ 4. Load the Whisper model { display-mode: "form" }
import whisper

if 'model_name' not in globals():
    print("❌ Please run the settings first.")
else:
    print(f"⏳ Loading '{model_name}'... please wait.")
    model = whisper.load_model(model_name, device=device)
    print("✅ Model ready.")

## 5. Transcribe

For every audio file in `sounds/` the notebook:
1. Transcribes it with the selected model, language and task
2. Saves the selected formats into `transcribes/` (UTF-8)
3. Records the time taken, and keeps going even if one file fails

Supported audio types: mp3, wav, m4a, flac, ogg, aac, wma, opus.

In [ ]:
#@title ▶ 5. Transcribe all audio files { display-mode: "form" }
import time
from tqdm.auto import tqdm
from whisper.utils import get_writer

if 'model' not in globals():
    print("❌ Please load the model first.")
else:
    AUDIO_EXTENSIONS = {".mp3", ".wav", ".m4a", ".flac", ".ogg", ".aac", ".wma", ".opus"}
    audio_files = sorted(p for p in sounds_dir.iterdir() if p.suffix.lower() in AUDIO_EXTENSIONS)

    def outputs_exist(audio_path):
        # True when every selected output already exists for this audio file.
        return all((transcribes_dir / f"{audio_path.stem}.{fmt}").exists() for fmt in output_formats)

    def transcribe_file(audio_path):
        options = dict(task=TASK, fp16=(device == "cuda"), verbose=False)
        if language_code:
            options["language"] = language_code
        if INITIAL_PROMPT.strip():
            options["initial_prompt"] = INITIAL_PROMPT.strip()
        return model.transcribe(str(audio_path), **options)

    def save_result(result, audio_path):
        for fmt in output_formats:
            writer = get_writer(fmt, str(transcribes_dir))
            writer(result, str(audio_path))

    summary = []

    if not audio_files:
        print(f"ℹ️  No audio files found in: {sounds_dir}")
        print("   Add audio files (or convert videos) and run this step again.")
    else:
        print(f"Found {len(audio_files)} audio file(s).\n")
        for audio_path in tqdm(audio_files, desc="Transcribing"):
            if SKIP_EXISTING and outputs_exist(audio_path):
                summary.append({"file": audio_path.name, "status": "skipped (already done)", "language": "-", "seconds": 0})
                continue

            start = time.time()
            try:
                result = transcribe_file(audio_path)
                save_result(result, audio_path)
                summary.append({"file": audio_path.name, "status": "done",
                                "language": result.get("language", "-"),
                                "seconds": round(time.time() - start, 1)})
            except Exception as error:
                summary.append({"file": audio_path.name, "status": f"failed: {error}", "language": "-", "seconds": 0})
                print(f"❌ {audio_path.name}: {error}")

        done = sum(1 for s in summary if s["status"] == "done")
        print(f"\n✅ Finished. {done} file(s) transcribed. Results are in: {transcribes_dir}")

## 6. Review the results

A summary table of what happened to each file, followed by a short preview of every `.txt` transcript.

In [ ]:
#@title ▶ 6a. Show the summary table { display-mode: "form" }
import pandas as pd
from IPython.display import display

if 'summary' in globals() and summary:
    display(pd.DataFrame(summary))
else:
    print("ℹ️  Nothing to show yet.")

In [ ]:
#@title ▶ 6b. Preview the transcripts { display-mode: "form" }
PREVIEW_CHARACTERS = 500  #@param {type:"integer"}

if 'transcribes_dir' not in globals():
    print("❌ Please run the settings first.")
else:
    txt_files = sorted(transcribes_dir.glob("*.txt"))
    if not txt_files:
        print("ℹ️  No .txt transcripts found yet.")
    for txt_file in txt_files:
        text = txt_file.read_text(encoding="utf-8")
        print(f"📄 {txt_file.name}  ({len(text):,} characters)")
        print(text[:PREVIEW_CHARACTERS] + (" ..." if len(text) > PREVIEW_CHARACTERS else ""))
        print("-" * 60)

## 7. Free GPU memory *(optional)*

Run this when you are finished, or before switching to a different model.

In [ ]:
#@title ▶ 7. Free up memory { display-mode: "form" }
import gc

if 'model' in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("🧹 Memory cleared.")

---
# 🧰 Standalone Tools

The two tools below are **independent** from Google Drive, the Whisper model, or Colab.

> **Running on your own computer?** The grey form fields only appear in Colab. In Jupyter or VS Code you will see them as normal lines at the top of the cell.

## Tool A — Extract Audio from Videos

Converts every video in a folder into an audio file. **No Google Drive and no Colab required.**

| Setting | Meaning |
|---|---|
| **VIDEO_FOLDER** | Folder that contains your videos. |
| **AUDIO_FOLDER** | Folder where the audio files will be saved (created automatically). |
| **AUDIO_FORMAT** | `mp3` (smaller) or `wav` (uncompressed). |
| **FOR_WHISPER** | Ticked = small mono 16 kHz files, ideal for transcription. Unticked = better sound quality for listening. |
| **INCLUDE_SUBFOLDERS** | Also convert videos inside subfolders (the folder structure is copied). |
| **OVERWRITE_EXISTING** | Tick to recreate audio files that already exist. |

**Requirement:** `ffmpeg`

In [ ]:
#@title ▶ Tool A. Extract audio from videos { display-mode: "form" }
VIDEO_FOLDER = "videos"  #@param {type:"string"}
AUDIO_FOLDER = "sounds"  #@param {type:"string"}
AUDIO_FORMAT = "mp3"  #@param ["mp3", "wav"]
FOR_WHISPER = True  #@param {type:"boolean"}
INCLUDE_SUBFOLDERS = False  #@param {type:"boolean"}
OVERWRITE_EXISTING = False  #@param {type:"boolean"}

import shutil
import subprocess
from pathlib import Path

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v", ".flv", ".wmv", ".mpg", ".mpeg", ".3gp"}

video_dir = Path(VIDEO_FOLDER.strip()).expanduser()
audio_dir = Path(AUDIO_FOLDER.strip()).expanduser()

if shutil.which("ffmpeg") is None:
    print("❌ ffmpeg is not installed (or not found), so videos cannot be converted.")
    print("   Install it, then run this cell again:")
    print("   • Windows : winget install ffmpeg   (then restart Jupyter)")
    print("   • macOS   : brew install ffmpeg")
    print("   • Linux   : sudo apt install ffmpeg")
    print("   • Colab   : run a cell with  !apt-get install -y ffmpeg")
elif not video_dir.is_dir():
    print(f"❌ Video folder not found: {video_dir.resolve()}")
    print("   Check the folder name in VIDEO_FOLDER.")
else:
    candidates = video_dir.rglob("*") if INCLUDE_SUBFOLDERS else video_dir.iterdir()
    videos = sorted(p for p in candidates if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)

    if not videos:
        print(f"ℹ️  No video files found in: {video_dir.resolve()}")
    else:
        audio_dir.mkdir(parents=True, exist_ok=True)
        print(f"Found {len(videos)} video(s) in {video_dir.resolve()}\n")
        converted = 0

        for video_path in videos:
            # Keep the same subfolder structure inside the audio folder.
            relative = video_path.relative_to(video_dir)
            audio_path = (audio_dir / relative).with_suffix("." + AUDIO_FORMAT)
            audio_path.parent.mkdir(parents=True, exist_ok=True)

            if audio_path.exists() and not OVERWRITE_EXISTING:
                print(f"⏭️  Already exists: {audio_path.name}")
                continue

            command = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(video_path), "-vn"]
            if FOR_WHISPER:
                command += ["-ac", "1", "-ar", "16000"]          # mono, 16 kHz
                if AUDIO_FORMAT == "mp3":
                    command += ["-b:a", "64k"]
            elif AUDIO_FORMAT == "mp3":
                command += ["-q:a", "2"]                          # high-quality mp3
            command.append(str(audio_path))

            print(f"🎬 Converting: {video_path.name}")
            result = subprocess.run(command, capture_output=True, text=True)
            if result.returncode != 0:
                print(f"   ❌ Could not convert this file: {result.stderr.strip()}")
            else:
                print(f"   ✅ Saved as {audio_path.name}")
                converted += 1

        print(f"\n✅ Done. {converted} audio file(s) created in: {audio_dir.resolve()}")

## Tool B — Convert Subtitle (.srt) Files to Text (.txt)

Every `.srt` file in a folder becomes its own `.txt` file containing only the spoken words (no numbers or timestamps).
It can run **on PC** or **in Google Colab** — choose with **RUN_ON**.

| Setting | Meaning |
|---|---|
| **RUN_ON** | `Local computer` or `Google Colab` (Colab will ask to connect your Drive). |
| **LOCAL_FOLDER** | Folder with the `.srt` files when running on your computer. |
| **COLAB_FOLDER** | Folder with the `.srt` files in Google Drive when running in Colab. |
| **OUTPUT_SUBFOLDER** | Subfolder for the new `.txt` files, so existing transcripts are not overwritten. Leave empty to save next to the `.srt` files. |
| **ONE_LINE_PER_SUBTITLE** | Ticked = one line per subtitle. Unticked = one continuous paragraph. |
| **OVERWRITE_EXISTING** | Tick to recreate `.txt` files that already exist. |

In [ ]:
#@title ▶ Tool B. Convert .srt files to .txt (local or Colab) { display-mode: "form" }
RUN_ON = "Local computer"  #@param ["Local computer", "Google Colab"]
LOCAL_FOLDER = "transcribes"  #@param {type:"string"}
COLAB_FOLDER = "/content/drive/MyDrive/Notebooks/transcribeWhisper/transcribes"  #@param {type:"string"}
OUTPUT_SUBFOLDER = "txt_from_srt"  #@param {type:"string"}
ONE_LINE_PER_SUBTITLE = False  #@param {type:"boolean"}
OVERWRITE_EXISTING = False  #@param {type:"boolean"}

import re
from pathlib import Path

def srt_to_text(content, one_line_per_subtitle):
    # Turns the content of an .srt file into plain text.
    content = content.replace("\ufeff", "").replace("\r\n", "\n").replace("\r", "\n")
    subtitles = []
    for block in re.split(r"\n\s*\n", content.strip()):
        lines = [line.strip() for line in block.split("\n") if line.strip()]
        # Remove the subtitle number and the timestamp line.
        if len(lines) >= 2 and lines[0].isdigit() and "-->" in lines[1]:
            lines = lines[2:]
        elif lines and "-->" in lines[0]:
            lines = lines[1:]
        # Remove formatting tags such as <i>...</i> or {\an8}.
        lines = [re.sub(r"\{\\an?\d+\}", "", re.sub(r"<[^>]+>", "", line)).strip() for line in lines]
        lines = [line for line in lines if line]
        if lines:
            subtitles.append(" ".join(lines))
    if one_line_per_subtitle:
        return "\n".join(subtitles) + "\n"
    return re.sub(r"\s+", " ", " ".join(subtitles)).strip() + "\n"

# --- Choose the folder depending on where the notebook is running ---
folder = None
if RUN_ON == "Google Colab":
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        folder = Path(COLAB_FOLDER.strip())
    except ImportError:
        print("❌ You chose 'Google Colab', but this notebook is not running in Colab.")
        print("   Change RUN_ON to 'Local computer' and run the cell again.")
else:
    folder = Path(LOCAL_FOLDER.strip()).expanduser()

if folder is not None:
    if not folder.is_dir():
        print(f"❌ Folder not found: {folder.resolve()}")
        print("   Check the folder name and try again.")
    else:
        srt_files = sorted(folder.glob("*.srt"))
        if not srt_files:
            print(f"ℹ️  No .srt files found in: {folder.resolve()}")
        else:
            out_dir = folder / OUTPUT_SUBFOLDER.strip() if OUTPUT_SUBFOLDER.strip() else folder
            out_dir.mkdir(parents=True, exist_ok=True)
            print(f"Found {len(srt_files)} .srt file(s).\n")
            created = 0
            for srt_path in srt_files:
                txt_path = out_dir / (srt_path.stem + ".txt")
                if txt_path.exists() and not OVERWRITE_EXISTING:
                    print(f"⏭️  Already exists: {txt_path.name}")
                    continue
                try:
                    content = srt_path.read_text(encoding="utf-8-sig", errors="replace")
                    txt_path.write_text(srt_to_text(content, ONE_LINE_PER_SUBTITLE), encoding="utf-8")
                    print(f"✅ {srt_path.name}  →  {txt_path.name}")
                    created += 1
                except Exception as error:
                    print(f"❌ {srt_path.name}: {error}")
            print(f"\n✅ Done. {created} text file(s) created in: {out_dir.resolve()}")